# Создание и заполнение данных БД Postgre

In [20]:
%pip install python-dotenv psycopg2-binary
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Ivan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 1.3 MB/s eta 0:00:07
   -- ------------------------------------- 0.5/9.7 MB 1.3 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/9.7 MB 853.1 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.7 MB 853.1 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/9.7 MB 777.9 kB/s eta 0:00:12
   ----- ---------------------------------- 1.3/9.7 MB 751.1 kB/s eta 0:00:12
   ----- ---------------------------------- 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Ivan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import psycopg2
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [5]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=Davydoff, DB=my_db_Davydoff, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [7]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [8]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [9]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


In [10]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура, специалитет, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [ ]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost",
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [22]:
query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
cursor.execute(query)
rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]
print(f"Колонки: {columns}\n")

for row in rows:
    print(row)

Колонки: ['courseid', 'userid', 'num_week', 's_all', 's_all_avg', 's_course_viewed', 's_course_viewed_avg', 's_q_attempt_viewed', 's_q_attempt_viewed_avg', 's_a_course_module_viewed', 's_a_course_module_viewed_avg', 's_a_submission_status_viewed', 's_a_submission_status_viewed_avg', 'namer_level', 'name_vatt', 'depart', 'name_osno', 'name_formopril', 'leveled', 'num_sem', 'kurs', 'date_vatt']

(75839, 32334, 6, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 5, 'Экзамен', 7, 1, 2, 2, 4, 3, datetime.date(2022, 6, 25))
(84328, 21129, 26, 0, 7.7143, 0, 1.3333, 0, 0.0, 0, 2.5714, 0, 1.5714, 5, 'Экзамен', 14, 1, 1, 1, 8, 5, datetime.date(2022, 6, 30))
(79913, 36408, 14, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 0, 0.0, 4, 'Экзамен', 11, 2, 2, 1, 2, 2, datetime.date(2022, 6, 27))
(79054, 24852, 6, 6, 6.0, 3, 3.0, 0, 0.0, 0, 0.0, 0, 0.0, 5, 'Экзамен', 5, 1, 1, 1, 6, 4, datetime.date(2022, 6, 9))
(71508, 29070, 8, 70, 29.6667, 20, 9.6667, 0, 0.0, 14, 5.0, 10, 3.6667, 3, 'Экзамен', 20, 1, 1, 1, 4, 3, datetime.da

In [33]:
import pandas as pd

query = "SELECT * FROM USER_LOGS ORDER BY RANDOM() LIMIT 10"
df = pd.read_sql_query(query, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_17232\3133113148.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,depart,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt
0,82135,29203,12,0,0.4286,0,0.2857,0,0.0000,0,...,0.0000,5,Экзамен,27,2,1,1,4,3,2022-06-17
1,72711,24527,8,0,18.3333,0,7.6667,0,0.0000,0,...,2.3333,5,Экзамен,2,1,1,1,6,4,2022-06-27
2,84080,33584,16,4,12.5455,4,3.3636,0,0.0000,0,...,1.3636,4,Экзамен,24,1,1,1,2,2,2022-06-17
3,81938,25138,18,0,0.0769,0,0.0769,0,0.0000,0,...,0.0000,5,Экзамен,16,1,1,1,6,4,2022-07-05
4,88721,31783,8,0,0.0000,0,0.0000,0,0.0000,0,...,0.0000,2,Экзамен,43,2,2,1,2,2,2022-06-29
5,84866,18761,23,0,0.5000,0,0.2778,0,0.0000,0,...,0.0556,5,Экзамен,15,1,2,2,10,6,2022-06-17
6,87396,27471,28,13,11.3913,4,3.2174,0,0.0000,3,...,2.3478,5,Экзамен,7,1,2,2,6,4,2022-06-23
7,84370,24890,16,50,54.0909,8,8.7273,0,0.2727,11,...,10.7273,4,Экзамен,23,1,1,1,6,4,2022-07-08
8,71884,28797,19,3,6.8571,2,3.7857,0,0.0000,0,...,0.0000,4,Экзамен,42,1,1,1,4,3,2022-06-22
9,76372,29453,18,0,5.2308,0,2.0000,0,0.0000,0,...,0.0000,5,Экзамен,43,1,1,1,4,3,2022-07-01


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [52]:
query = """
    SELECT
        COUNT(DEPT.depart)
    FROM
        (
        SELECT
            depart,
            COUNT(courseid)
        FROM USER_LOGS
        GROUP BY depart
        HAVING COUNT(courseid) <> 0
        ) AS DEPT
    
"""

query2 = """
    SELECT 
        COUNT(DISTINCT depart) AS departments_count
    FROM USER_LOGS
    WHERE courseid IS NOT NULL
"""

df = pd.read_sql_query(query2, conn)
display(df)

C:\Users\Ivan\AppData\Local\Temp\ipykernel_17232\767811180.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query2, conn)


,departments_count
0,43


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |